## Step 4: Driver Verification

Now that we have encapsulated the hardware logic into the `MotorDriver` and `MPU6050Driver` classes, we will test the **Automatic Mapping** feature of PYNQ. 

If our `bindto` strings were correct, PYNQ will no longer treat the IPs as generic memory blocks; it will treat them as our custom Python objects.

In [1]:
from pynq import Overlay
from balancin.drivers import MotorDriver, MPU6050Driver

# 1. Load the overlay
ol = Overlay("balancin/bitstreams/balancin.bit")

# 2. MANUALLY OVERRIDE the default drivers with your custom ones
# We pass the IP's metadata (from ol.ip_dict) to our class constructors
ol.axi_gpio_0 = MotorDriver(ol.ip_dict['axi_gpio_0'])
ol.axi_iic_0 = MPU6050Driver(ol.ip_dict['axi_iic_0'])

# 3. Verify the types again - they should now show your custom classes
print(f"Type of axi_gpio_0: {type(ol.axi_gpio_0)}")
print(f"Type of axi_iic_0: {type(ol.axi_iic_0)}")

Type of axi_gpio_0: <class 'balancin.drivers.MotorDriver'>
Type of axi_iic_0: <class 'balancin.drivers.MPU6050Driver'>


In [2]:
# Test the sensor
angle = ol.axi_iic_0.get_angle()
print(f"Current sensor angle: {angle:.2f} degrees")

# Test the motor using our new scale (0 is stop, 1023 is max)
# Remember: The driver handles the inverse logic for us!
print("Starting motor at speed 200...")
ol.axi_gpio_0.set_speed(200)

import time
time.sleep(2)

print("Stopping motor...")
ol.axi_gpio_0.stop()

Current sensor angle: -26.15 degrees
Starting motor at speed 200...
Stopping motor...


In [1]:
from balancin.overlay import BalancinOverlay
ol = BalancinOverlay()

print(f"Motor class: {type(ol.motor)}")
print(f"Sensor class: {type(ol.sensor)}")

# Verify it actually works
print(f"Current angle: {ol.sensor.get_angle()}")

Motor class: <class 'balancin.drivers.MotorDriver'>
Sensor class: <class 'balancin.drivers.MPU6050Driver'>
Current angle: -26.445853641806593


In [5]:
from balancin.pid import PID

# Test: If error is 10 and Kp is 2, output should be 20
test_pid = PID(kp=2.0, ki=0, kd=0)
result = test_pid.compute(setpoint=20, measured_value=10, dt=0.02)
print(f"PID Test Result: {result}") # Should be 20.0

PID Test Result: 20.0


In [7]:
from balancin.overlay import BalancinOverlay
from balancin.balancer import Balancer

ol = BalancinOverlay()
bal = Balancer(ol)

# This returns immediately!
bal.start()

# You can now change things while it runs:
bal.set_params(kp=4.2) 

# Stop it when you are done
bal.stop()

Balancer thread started.
Balancer stopped and motor safety shutdown complete.
